In [ ]:
from transformers import ViTImageProcessor, ViTForImageClassification
from datasets import load_dataset
from PIL import Image
import requests
import torch
import PIL
from operator import itemgetter
import numpy as np

In [15]:
device = "cpu"
# ViT requires flash attention which is not supported on a lot of GPUs, and not on Windows
# see: https://github.com/pytorch/pytorch/issues/108175

In [16]:
print(torch.cuda_version)

12.1


In [17]:
test_dataset = load_dataset("cifar10", split="test")

In [18]:
processor: ViTImageProcessor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")
model = ViTForImageClassification.from_pretrained("phuong-tk-nguyen/vit-base-patch16-224-finetuned-cifar10").to(
    device
)  # Apache License 2.0

In [19]:
test_dataset

Dataset({
    features: ['img', 'label'],
    num_rows: 10000
})

In [20]:
get_img = itemgetter("img")
imgs = list(map(get_img, test_dataset))

In [21]:
inputs = processor.preprocess(images=imgs, return_tensors="pt")
inputs

{'pixel_values': tensor([[[[ 0.2392,  0.2392,  0.2392,  ..., -0.0902, -0.0902, -0.0902],
          [ 0.2392,  0.2392,  0.2392,  ..., -0.0902, -0.0902, -0.0902],
          [ 0.2392,  0.2392,  0.2392,  ..., -0.0902, -0.0902, -0.0902],
          ...,
          [-0.5765, -0.5765, -0.5765,  ..., -0.8353, -0.8353, -0.8353],
          [-0.5765, -0.5765, -0.5765,  ..., -0.8353, -0.8353, -0.8353],
          [-0.5765, -0.5765, -0.5765,  ..., -0.8353, -0.8353, -0.8353]],

         [[-0.1216, -0.1216, -0.1216,  ..., -0.3333, -0.3333, -0.3333],
          [-0.1216, -0.1216, -0.1216,  ..., -0.3333, -0.3333, -0.3333],
          [-0.1216, -0.1216, -0.1216,  ..., -0.3333, -0.3333, -0.3333],
          ...,
          [-0.1608, -0.1608, -0.1608,  ..., -0.4745, -0.4745, -0.4745],
          [-0.1608, -0.1608, -0.1608,  ..., -0.4745, -0.4745, -0.4745],
          [-0.1608, -0.1608, -0.1608,  ..., -0.4745, -0.4745, -0.4745]],

         [[-0.6157, -0.6157, -0.6157,  ..., -0.7412, -0.7412, -0.7412],
          [-0

In [22]:
inputs = inputs["pixel_values"]
type(inputs)

torch.Tensor

In [23]:
outputs = model(pixel_values=inputs[:10].to(device))
predicted_labels = outputs.logits.argmax(-1)
predicted_labels

tensor([3, 8, 8, 0, 6, 6, 1, 6, 3, 1])

In [24]:
label_ids = test_dataset["label"][:10]
for predicted_label, label_id in zip(predicted_labels, label_ids):
    print(model.config.id2label[predicted_label.item()], "|", model.config.id2label[label_id])

Cat | Cat
Ship | Ship
Ship | Ship
Airplane | Airplane
Frog | Frog
Frog | Frog
Automobile | Automobile
Frog | Frog
Cat | Cat
Automobile | Automobile


In [26]:
# REDUCING the batch size to 10 so that no weird memory error occurs
batch_size = 10
predicted_labels = []
for batch in range(0, len(test_dataset), batch_size):
    inputs = processor.preprocess(images=imgs[batch : batch + batch_size], return_tensors="pt")
    inputs = inputs["pixel_values"]
    outputs = model(pixel_values=inputs.to(device))
    predicted_labels.extend(outputs.logits.argmax(-1))

predicted_labels

[tensor(3),
 tensor(8),
 tensor(8),
 tensor(0),
 tensor(6),
 tensor(6),
 tensor(1),
 tensor(6),
 tensor(3),
 tensor(1),
 tensor(0),
 tensor(9),
 tensor(5),
 tensor(7),
 tensor(9),
 tensor(8),
 tensor(5),
 tensor(7),
 tensor(8),
 tensor(6),
 tensor(7),
 tensor(0),
 tensor(4),
 tensor(9),
 tensor(5),
 tensor(2),
 tensor(4),
 tensor(0),
 tensor(9),
 tensor(6),
 tensor(6),
 tensor(5),
 tensor(4),
 tensor(5),
 tensor(9),
 tensor(2),
 tensor(4),
 tensor(9),
 tensor(9),
 tensor(5),
 tensor(4),
 tensor(6),
 tensor(5),
 tensor(6),
 tensor(0),
 tensor(9),
 tensor(3),
 tensor(9),
 tensor(7),
 tensor(6),
 tensor(9),
 tensor(8),
 tensor(0),
 tensor(3),
 tensor(8),
 tensor(8),
 tensor(7),
 tensor(7),
 tensor(4),
 tensor(6),
 tensor(7),
 tensor(5),
 tensor(6),
 tensor(3),
 tensor(6),
 tensor(2),
 tensor(1),
 tensor(2),
 tensor(3),
 tensor(7),
 tensor(2),
 tensor(6),
 tensor(8),
 tensor(8),
 tensor(0),
 tensor(2),
 tensor(9),
 tensor(3),
 tensor(3),
 tensor(8),
 tensor(8),
 tensor(1),
 tensor(1),
 ten

In [28]:
from sklearn import metrics

In [32]:
print(
    metrics.classification_report(
        test_dataset["label"], predicted_labels, target_names=model.config.id2label.values(), digits=4
    )
)

              precision    recall  f1-score   support

    Airplane     0.9890    0.9890    0.9890      1000
  Automobile     0.9802    0.9890    0.9846      1000
        Bird     0.9841    0.9930    0.9886      1000
         Cat     0.9752    0.9450    0.9599      1000
        Deer     0.9840    0.9870    0.9855      1000
         Dog     0.9596    0.9750    0.9673      1000
        Frog     0.9940    0.9950    0.9945      1000
       Horse     0.9950    0.9920    0.9935      1000
        Ship     0.9880    0.9920    0.9900      1000
       Truck     0.9859    0.9780    0.9819      1000

    accuracy                         0.9835     10000
   macro avg     0.9835    0.9835    0.9835     10000
weighted avg     0.9835    0.9835    0.9835     10000

